In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

import sys
import subprocess

packages = [
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *packages
])

print("Done. Restart Kaggle Session 1 lần rồi chạy lại từ Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.7 MB/s eta 0:00:00
Done. Restart Kaggle Session 1 lần rồi chạy lại từ Cell 2.


In [2]:
# ============================================================
# CELL 2 — ENVIRONMENT CHECK
# ============================================================

import os

# Chỉ dùng GPU 0 -> tránh DataParallel với model 4-bit
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import transformers
import peft
import trl
import bitsandbytes
import accelerate

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("Python       :", __import__("sys").version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("TRL          :", trl.__version__)
print("bnb          :", bitsandbytes.__version__)
print("Accelerate   :", accelerate.__version__)

print("\nGPU")
print("=" * 70)

assert torch.cuda.is_available(), "CUDA GPU không khả dụng!"

print("GPU :", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

# T4 không dùng BF16
assert "T4" in torch.cuda.get_device_name(0), \
    f"Notebook này được thiết kế cho T4, nhưng GPU hiện tại là {torch.cuda.get_device_name(0)}"

print("\nOK: Single GPU + FP16 + T4")

ENVIRONMENT
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
PEFT         : 0.19.1
TRL          : 1.10.0
bnb          : 0.50.1
Accelerate   : 1.13.0

GPU
GPU : Tesla T4
VRAM: 14.56 GB

OK: Single GPU + FP16 + T4


In [3]:
# ============================================================
# CELL 3 — IMPORTS & CONFIG
# ============================================================

import os
import gc
import glob
import json
import inspect
import random

import torch

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer


MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

DATA_DIR = "/kaggle/input/datasets/lionelhuansi/text2cypher-sft-37k/train_kd_37k"
DATA_FILE = os.path.join(DATA_DIR, "train_kd_37k.json")

OUTPUT_DIR = "/kaggle/working/qwen2.5_1.5b_kd_adapter"
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
MAX_LENGTH = 1280

# T4: ưu tiên an toàn VRAM
MICRO_BATCH = 4
GRAD_ACCUM = 8

EPOCHS = 2
LEARNING_RATE = 2e-4

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("MODEL :", MODEL_ID)
print("DATA  :", DATA_FILE)
print("OUT   :", FINAL_DIR)

MODEL : unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
DATA  : /kaggle/input/datasets/lionelhuansi/text2cypher-sft-37k/train_kd_37k/train_kd_37k.json
OUT   : /kaggle/working/qwen2.5_1.5b_kd_adapter/final_adapter


In [4]:
# ============================================================
# CELL 4 — LOAD DATASET
# ============================================================

assert os.path.exists(DATA_FILE), f"Không tìm thấy:\n{DATA_FILE}"

with open(DATA_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("Number of raw samples:", len(raw_data))

# Kiểm tra format
sample = raw_data[0]

print("\nDataset keys:")
print(list(sample.keys()))

print("\nFirst sample:")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:4000])


# Dataset của bạn đã có sẵn:
#   prompt
#   completion
#   text
#
# Trong đó "text" = prompt + completion
#
# => Không cần tự ghép lại.


assert "text" in sample, \
    "Dataset không có field 'text'."

assert "prompt" in sample, \
    "Dataset không có field 'prompt'."

assert "completion" in sample, \
    "Dataset không có field 'completion'."


# Chỉ giữ field text cho SFT
dataset = Dataset.from_list([
    {
        "text": str(item["text"])
    }
    for item in raw_data
])

print("\n" + "=" * 70)
print("PROCESSED DATASET")
print("=" * 70)

print(dataset)
print("Columns:", dataset.column_names)

print("\nFirst training example:")
print(dataset[0]["text"][:5000])

Number of raw samples: 37408

Dataset keys:
['prompt', 'completion', 'text']

First sample:
{
  "prompt": "### System:\nYou are an expert Graph Database AI. Translate the natural language question into a Cypher query.\nYou MUST return a single raw JSON object matching this EXACT schema structure:\n{\n  \"instance_extraction\": [\n    {\"entity_or_property\": \"string\", \"value\": \"string or number\", \"entity_type\": \"string\"}\n  ],\n  \"relation_mapping\": [\n    {\"source_node\": \"string\", \"relation\": \"string\", \"target_node\": \"string\", \"direction\": \"OUTGOING|INCOMING\"}\n  ],\n  \"validation_check\": {\"status\": \"PASS\", \"description\": \"string\"},\n  \"cypher\": \"string\"\n}\n\n### Input:\nOntology/Schema:\nNode properties:\n- **Country**\n  - `location`: POINT \n  - `code`: STRING Example: \"AFG\"\n  - `name`: STRING Example: \"Afghanistan\"\n  - `tld`: STRING Example: \"AF\"\n- **Filing**\n  - `begin`: DATE_TIME Min: 2000-02-08T00:00:00Z, Max: 2017-09-05T00:0

In [5]:
# ============================================================
# CELL 5 — TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Vocab size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)


# Kiểm tra độ dài token của dataset
sample_text = dataset[0]["text"]

tokens = tokenizer(
    sample_text,
    add_special_tokens=True,
    truncation=False,
)

print("\nFirst sample token count:", len(tokens["input_ids"]))

if len(tokens["input_ids"]) > MAX_LENGTH:
    print(
        f"WARNING: sample dài hơn MAX_LENGTH={MAX_LENGTH}, "
        "sẽ bị truncate khi training."
    )
else:
    print(f"Sample nằm trong MAX_LENGTH={MAX_LENGTH}.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Tokenizer loaded.
Vocab size: 151665
EOS token: <|im_end|>
PAD token: <|vision_pad|>

First sample token count: 1080
Sample nằm trong MAX_LENGTH=2048.


In [6]:
# ============================================================
# CELL 6 — LOAD 4-BIT MODEL
# ============================================================

gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",

    # T4 -> FP16, KHÔNG dùng BF16
    bnb_4bit_compute_dtype=torch.float16,

    # Tiết kiệm thêm VRAM
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,

    # Chỉ GPU 0
    device_map={"": 0},

    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.config.use_cache = False

print("Model loaded successfully.")
print("Device:", next(model.parameters()).device)

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Model loaded successfully.
Device: cuda:0


In [7]:
# ============================================================
# CELL 7 — LoRA CONFIG
# ============================================================

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],

    bias="none",
    use_rslora=False,
    use_dora=False,
)

print("LoRA configured.")
print(peft_config)

LoRA configured.
LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'v_proj', 'q_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [8]:
# ============================================================
# CELL 8 — BUILD SFT CONFIG COMPATIBLY
# ============================================================

sft_sig = inspect.signature(SFTConfig)
sft_params = sft_sig.parameters

kwargs = dict(
    output_dir=OUTPUT_DIR,

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,

    logging_steps=10,
    save_steps=250,
    save_total_limit=3,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    report_to="none",

    seed=SEED,

    dataloader_num_workers=2,

    remove_unused_columns=False,
)

# TRL mới
if "max_length" in sft_params:
    kwargs["max_length"] = MAX_LENGTH

# TRL cũ
elif "max_seq_length" in sft_params:
    kwargs["max_seq_length"] = MAX_LENGTH

# Dataset đã có text
if "dataset_text_field" in sft_params:
    kwargs["dataset_text_field"] = "text"

# Giảm VRAM
if "packing" in sft_params:
    kwargs["packing"] = False


training_args = SFTConfig(**kwargs)

print("SFTConfig created successfully.")
print(training_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig created successfully.
SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
ena

In [9]:
# ============================================================
# CELL 9 — CREATE SFT TRAINER
# ============================================================

trainer_sig = inspect.signature(SFTTrainer)
trainer_params = trainer_sig.parameters

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=dataset,

    # Để TRL/PEFT tự inject LoRA
    peft_config=peft_config,
)

# TRL mới
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer

# TRL cũ
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Trainer created successfully.")

Adding EOS to train dataset:   0%|          | 0/37408 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/37408 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/37408 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/37408 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/37408 [00:00<?, ? examples/s]

Trainer created successfully.


In [10]:
# ============================================================
# CELL 10 — RESUME CHECKPOINT
# ============================================================

# Sau khi SFTTrainer tạo PEFT model
trainer.model.print_trainable_parameters()


def find_latest_checkpoint(output_dir):
    checkpoints = glob.glob(
        os.path.join(output_dir, "checkpoint-*")
    )

    valid = []

    for path in checkpoints:
        name = os.path.basename(path)

        try:
            step = int(name.split("-")[-1])
            valid.append((step, path))
        except ValueError:
            pass

    if not valid:
        return None

    valid.sort(key=lambda x: x[0])

    return valid[-1][1]


latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)

print("\nLatest checkpoint:")

if latest_checkpoint:
    print(latest_checkpoint)
else:
    print("None -> training from scratch")

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Latest checkpoint:
None -> training from scratch


In [ ]:
# ============================================================
# CELL 11 — TRAIN WITH AUTOMATIC CHECKPOINT RESUME & FP32 FIX
# ============================================================

import os
import shutil
import glob
import torch

print("=" * 70)
print("CHECKING TRAINABLE PARAMETERS")
print("=" * 70)

trainable = []
dtype_count = {}

for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        trainable.append((name, param))
        dtype_count[str(param.dtype)] = (
            dtype_count.get(str(param.dtype), 0) + 1
        )

print("\nBefore dtype fix:")
for dtype, count in dtype_count.items():
    print(f"  {dtype}: {count} parameters")

# Force trainable LoRA parameters to FP32 for T4 + GradScaler safety
converted = 0
for name, param in trainable:
    if param.dtype != torch.float32:
        param.data = param.data.float()
        converted += 1

print(f"\nConverted {converted} trainable tensors to FP32.")

# Check BF16 trainable parameters
bf16_trainable = [
    name for name, param in trainer.model.named_parameters()
    if param.requires_grad and param.dtype == torch.bfloat16
]
if bf16_trainable:
    raise RuntimeError(
        "Vẫn còn trainable BF16 parameters:\n" + "\n".join(bf16_trainable[:20])
    )

print("\nOK: No trainable BF16 parameters.")

# ------------------------------------------------------------
# CHECKPOINT FINDER (Searches both Working and Input)
# ------------------------------------------------------------
def find_latest_checkpoint(output_dir):
    patterns = [
        os.path.join(output_dir, "checkpoint-*"),
        "/kaggle/input/**/checkpoint-*",
        "/kaggle/input/**/qwen2.5_1.5b_kd_adapter/checkpoint-*"
    ]
    
    checkpoints = []
    for p in patterns:
        checkpoints.extend(glob.glob(p, recursive=True))

    valid = []
    for path in checkpoints:
        name = os.path.basename(path)
        try:
            step = int(name.split("-")[-1])
            if os.path.isdir(path) and len(os.listdir(path)) > 0:
                valid.append((step, path))
        except ValueError:
            pass

    if not valid:
        return None

    valid.sort(key=lambda x: x[0])
    best_step, best_path = valid[-1]
    
    working_target = os.path.join(output_dir, f"checkpoint-{best_step}")
    if not os.path.exists(working_target) or len(os.listdir(working_target)) == 0:
        print(f"Copying checkpoint-{best_step} from Input ({best_path}) to Working ({working_target})...")
        if os.path.exists(working_target):
            shutil.rmtree(working_target)
        shutil.copytree(best_path, working_target)
    
    return working_target

latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)

print("\n" + "=" * 70)
if latest_checkpoint:
    print("RESUMING TRAINING FROM CHECKPOINT")
    print("=" * 70)
    print("Checkpoint:", latest_checkpoint)
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("START TRAINING FROM SCRATCH")
    print("=" * 70)
    trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


CHECKING TRAINABLE PARAMETERS

Before dtype fix:
  torch.bfloat16: 224 parameters

Converted 224 trainable tensors to FP32.

After dtype fix:
  torch.float32: 224 parameters

OK: No trainable BF16 parameters.

START TRAINING FROM SCRATCH


Step,Training Loss
10,1.555577
20,1.610294
30,1.570670
40,1.433262
50,1.388630
60,1.289630
70,1.082397


In [ ]:
# ============================================================
# CELL 12 — SAVE FINAL LORA ADAPTER
# ============================================================

import os

os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print("Adapter saved to:")
print(FINAL_DIR)

print("\nFiles:")
for f in sorted(os.listdir(FINAL_DIR)):
    print(" -", f)

In [ ]:
# ============================================================
# CELL 13 — QUICK TEST
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

SYSTEM_PROMPT = """You are an expert Graph Database AI. Translate the natural language question into a Cypher query.
You MUST return a single raw JSON object matching this EXACT schema structure:
{
  "instance_extraction": [
    {"entity_or_property": "string", "value": "string or number", "entity_type": "string"}
  ],
  "relation_mapping": [
    {"source_node": "string", "relation": "string", "target_node": "string", "direction": "OUTGOING|INCOMING"}
  ],
  "validation_check": {"status": "PASS", "description": "string"},
  "cypher": "string"
}"""

tokenizer_test = AutoTokenizer.from_pretrained(FINAL_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

test_model = PeftModel.from_pretrained(
    base_model,
    FINAL_DIR,
)

test_question = "Which 3 countries have the most entities linked as beneficiaries in filings?"

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": test_question,
    },
]

prompt = tokenizer_test.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer_test(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=1.0,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

print("=" * 70)
print("PREDICTED CYPHER OUTPUT:")
print("=" * 70)
print(tokenizer_test.decode(
    generated,
    skip_special_tokens=True
))